# v121_france_from_v107 — v110 for India and the US, v107 for France

| Field | Value |
|---|---|
| **Version** | `v121_france_from_v107` |
| **Plan group** | INT (diagnostic submission) |
| **Parent version** | v110 (India, US) and v107 (France) |
| **Author** | M1 rajaguru2004 |
| **Date** | 2026-09-26 |
| **Status** | shortlisted |

v110 scored 0.966 public, the same as v107, although the tight mock estimated 0.9731 (it had
predicted v107 to within 0.0001). The mock has no French entity (France is 15 % of the test's
S1, absent from train). Against v107, v110 leaves 87.8 % of Indian and 86.4 % of US rows
unchanged and mostly adds matches there, but only 78.6 % of French rows: it drops matches for
5.4 % of French S1 (1.8 % India, 1.3 % US) and replaces them for 1.1 % (0.2 %).

## 1. Hypothesis

* **Change vs parent:** the French rows of both files come from v107; India and US stay v110.
* **Why:** if v110's India / US gains are real (mock +0.0102 / +0.0040), a flat public score
  means its French rows lost about 0.04 F0.5; v107's French rows would then lift the public
  score to about 0.972.
* **Check:** the public score. Above 0.969: France is v110's problem and later versions keep a
  France-safe path; about 0.966: the mock's gain did not reach the test for India and the US.

## 2. Setup

Both submissions' files are read back as (S1, pool id) pairs; each S1 row is taken from the
version of its country.

In [1]:
import shutil
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd

from entity_resolution import config as C
from entity_resolution.data import load_source
from entity_resolution.submission import write_pairs
from entity_resolution.tracking import log_result

EXP_DIR = C.EXPERIMENTS / "v121_france_from_v107"
SUB = C.ROOT / "submissions"
FRANCE_FROM, REST_FROM = "v107", "v110"
t0 = time.time()
s1 = load_source("test", 1, C.DATASET, columns=[C.ENTITY_ID, C.COUNTRY])
country_of = s1.set_index(C.ENTITY_ID)[C.COUNTRY]


def pairs_of(path: Path) -> pd.DataFrame:
    """(S1 id, pool id) pairs of a submission file, one row per listed pool id."""
    f = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)
    col = f.columns[1]
    f = f.assign(**{col: f[col].str.split(",")}).explode(col)
    f = f[f[col] != ""]
    return pd.DataFrame({C.S1_ID: f.iloc[:, 0].to_numpy(), C.ENTITY_ID: f[col].to_numpy()})


def compose(name: str) -> pd.DataFrame:
    """French rows from FRANCE_FROM, every other country's rows from REST_FROM."""
    fr, rest = pairs_of(SUB / FRANCE_FROM / name), pairs_of(SUB / REST_FROM / name)
    is_fr = lambda d: (d[C.S1_ID].map(country_of) == "France").to_numpy()   # noqa: E731
    return pd.concat([rest[~is_fr(rest)], fr[is_fr(fr)]], ignore_index=True)


matches, cands = compose(C.MATCHING_FILE), compose(C.CANDIDATE_FILE)
print(f"{len(matches):,} matched pairs, {len(cands):,} candidate pairs")

5,786,367 matched pairs, 8,795,496 candidate pairs


## 3. Write and validate

`write_pairs` writes both files (one row per test S1, in the test's order) to `output/`; they
are copied to `submissions/v121/` and both validators run.

In [2]:
match_path, cand_path = write_pairs(matches, cands, s1[C.ENTITY_ID].tolist(), C.OUTPUT)
dest = SUB / "v121"
dest.mkdir(parents=True, exist_ok=True)
for p in (match_path, cand_path):
    shutil.copy2(p, dest / p.name)
out = subprocess.run([sys.executable, "-m", "entity_resolution.submission", "--output-dir",
                      str(C.OUTPUT), "--check-ids"], capture_output=True, text=True)
print(out.stdout[-1500:], out.stderr[-1500:])
out = subprocess.run([sys.executable, str(C.OFFICIAL_VALIDATOR), "--matching", str(match_path),
                      "--candidate", str(cand_path), "--test-dir", str(C.DATASET / "test")],
                     capture_output=True, text=True)
print(out.stdout[-2000:], out.stderr[-1500:])
by = matches[C.S1_ID].map(country_of)
n_s1 = country_of.value_counts()
pd.DataFrame({"matched_share": matches.drop_duplicates(C.S1_ID)[C.S1_ID].map(country_of)
              .value_counts() / n_s1, "matches_per_s1": by.value_counts() / n_s1})

PASS
 


ML Challenge 2026 — submission validator
  test dir: /home/suryaguru/StudioProjects/aws/business_entity_resolution/dataset/student_resource/dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (100198 empty, 1632346 non-empty).
  candidate_pairs.tsv: 1732544 rows (24032 empty, 1708512 non-empty).

PASS — no blocking issues found. Safe to submit.
 


,matched_share,matches_per_s1
source1_entity_id,,
India,0.940737,3.296409
US,0.942726,3.428919
France,0.945204,3.247556


## 4. Log the result

No model is trained: the mock score is v110's (the mock has no French entity); the public score
decides.

In [3]:
v110 = __import__("json").loads((C.EXPERIMENTS / "v110_m3_features" / "metrics.json").read_text())
row = log_result(
    EXP_DIR, change="v110's rows for India and the US, v107's rows for France (diagnostic)",
    group="INT", mock_f05=float(v110["mock_f05"]),
    notes="tests whether v110's flat public score (0.966 vs est 0.9731) comes from France",
    metrics={"france_from": FRANCE_FROM, "rest_from": REST_FROM,
             "matched_pairs": len(matches), "candidate_pairs": len(cands),
             "seconds": round(time.time() - t0, 1)},
    owner="M1", parent="v110", decision="")
row

{'version': 'v121',
 'date': '2026-09-26',
 'group': 'INT',
 'change': "v110's rows for India and the US, v107's rows for France (diagnostic)",
 'local_f05': '',
 'mock_f05': '0.9817',
 'cand_recall': '',
 'public_f05': '',
 'commit': 'a1c00d6',
 'notes': "tests whether v110's flat public score (0.966 vs est 0.9731) comes from France",
 'owner': 'M1',
 'parent': 'v110',
 'decision': ''}

## 5. Conclusion

Written after the upload from the public score.